In [1]:
from typing import List, Tuple, Dict, Optional
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
import math

In [2]:
# Data processing utilities

In [5]:
class DataProcessor:
    def __init__(self, ratings_path: str = "ml-100k/u.data", items_path: str = "ml-100k/u.item"):
        self.ratings_path = ratings_path
        self.items_path = items_path
        self.ratings = None
        self.items = None
        self.user_item_matrix = None

        self.user_index_to_id = None
        self.movie_index_to_id = None

    def load(self):
        # MovieLens 100k format: user_id, movie_id, rating, timestamp
        cols = ["user_id", "movie_id", "rating", "timestamp"]
        self.ratings = pd.read_csv(self.ratings_path, sep="\\t", names=cols, engine="python")

        # u.item has many fields; the first two are movie id and title
        item_cols = ["movie_id", "title"] + [f"f{i}" for i in range(22)]
        self.items = pd.read_csv(self.items_path, sep="|", encoding='latin-1', names=item_cols, usecols=[0,1], engine="python")
        self.items['movie_id'] = self.items['movie_id'].astype(int)

        return self

    def preprocess(self, min_user_ratings: int = 5, min_movie_ratings: int = 5, dedupe: bool = True):
        r = self.ratings
        if dedupe:
            before = len(r)
            r = r.drop_duplicates(subset=["user_id","movie_id"]).copy()
            # If multiple ratings exist, keep the latest
            # (for explicit dedupe we kept first occurrence after drop_duplicates)
            print(f"Dropped {before - len(r)} duplicate rating rows.")

        # Remove missing values
        r = r.dropna(subset=["user_id", "movie_id", "rating"]).copy()

        # Filter inactive users and rare movies
        user_counts = r['user_id'].value_counts()
        keep_users = user_counts[user_counts >= min_user_ratings].index
        movie_counts = r['movie_id'].value_counts()
        keep_movies = movie_counts[movie_counts >= min_movie_ratings].index

        r = r[r['user_id'].isin(keep_users) & r['movie_id'].isin(keep_movies)].copy()

        self.ratings = r

        # Build sparse user-item matrix (rows = users, cols = movies)
        users = sorted(r['user_id'].unique())
        movies = sorted(r['movie_id'].unique())
        user_map = {u:i for i,u in enumerate(users)}
        movie_map = {m:j for j,m in enumerate(movies)}

        rows = r['user_id'].map(user_map).to_numpy()
        cols = r['movie_id'].map(movie_map).to_numpy()
        data = r['rating'].to_numpy()

        mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(users), len(movies)))

        # Save useful metadata
        self.user_index_to_id = users
        self.movie_index_to_id = movies
        self.user_id_to_index = user_map
        self.movie_id_to_index = movie_map
        self.user_item_matrix = mat

        return self

    def get_user_vector(self, user_id: int) -> Optional[sparse.csr_matrix]:
        idx = self.user_id_to_index.get(user_id)
        if idx is None:
            return None
        return self.user_item_matrix.getrow(idx)

In [9]:
# Similarity helpers

def cosine_sim_sparse(matrix: sparse.csr_matrix) -> np.ndarray:
    # returns dense similarity matrix (careful for very large matrices)
    arr = matrix.toarray()
    sim = cosine_similarity(arr)
    return sim

def pearson_between_items(matrix: sparse.csr_matrix) -> np.ndarray:
    # Compute Pearson correlation between columns (items)
    A = matrix.toarray()
    # center columns
    A_centered = A - A.mean(axis=0, keepdims=True)
    denom = np.sqrt((A_centered**2).sum(axis=0))
    denom[denom == 0] = 1e-9
    corr = (A_centered.T @ A_centered) / (denom[:,None] * denom[None,:])
    return corr

In [10]:
#Collaborative Filtering

In [11]:
class CollaborativeFiltering:
    def __init__(self, processor: DataProcessor):
        self.dp = processor
        self.user_sim = None
        self.item_sim = None

    def fit_item_based(self, method: str = 'cosine'):
        # item-based similarity on transposed matrix
        M = self.dp.user_item_matrix
        if method == 'cosine':
            # compute on item vectors (columns): convert to (n_items, n_users)
            sim = cosine_similarity(M.T.toarray())
        elif method == 'pearson':
            sim = pearson_between_items(M)
        else:
            raise ValueError('unknown method')
        self.item_sim = sim
        return self

    def fit_user_based(self):
        M = self.dp.user_item_matrix
        self.user_sim = cosine_similarity(M.toarray())
        return self

    def predict_item_based_for_user(self, user_id: int, top_k: int = 10) -> List[Tuple[int, float]]:
        idx = self.dp.user_id_to_index.get(user_id)
        if idx is None:
            return []
        user_row = self.dp.user_item_matrix.getrow(idx).toarray().ravel()
        # scores = item_sim @ user_ratings / sum(abs(similarities))
        sim = self.item_sim
        scores = sim.dot(user_row)
        denom = np.abs(sim).sum(axis=1)
        denom[denom==0] = 1e-9
        scores = scores / denom

        # filter already-watched
        watched = set(np.where(user_row>0)[0])
        candidates = [(i, s) for i,s in enumerate(scores) if i not in watched]
        candidates.sort(key=lambda x: x[1], reverse=True)
        top = candidates[:top_k]
        # convert indices to movie ids
        return [(self.dp.movie_index_to_id[i], float(score)) for i,score in top]

    def predict_user_based_for_user(self, user_id: int, top_k: int = 10, k_similar_users: int = 20) -> List[Tuple[int,float]]:
        idx = self.dp.user_id_to_index.get(user_id)
        if idx is None or self.user_sim is None:
            return []
        user_row = self.dp.user_item_matrix.getrow(idx).toarray().ravel()
        # find similar users
        sims = self.user_sim[idx]
        top_users = np.argsort(sims)[::-1][1:k_similar_users+1]
        # weighted sum of their ratings
        neighbor_ratings = self.dp.user_item_matrix[top_users].toarray()
        weights = sims[top_users]
        weighted = neighbor_ratings.T.dot(weights)
        denom = np.sum(np.abs(weights))
        if denom==0: denom = 1e-9
        scores = weighted / denom

        watched = set(np.where(user_row>0)[0])
        candidates = [(i, s) for i,s in enumerate(scores) if i not in watched]
        candidates.sort(key=lambda x: x[1], reverse=True)
        top = candidates[:top_k]
        return [(self.dp.movie_index_to_id[i], float(score)) for i,score in top]

In [12]:
# Matrix Factorization (lightweight via TruncatedSVD)

In [13]:
class MatrixFactorization:
    def __init__(self, dp: DataProcessor, n_components: int = 20):
        self.dp = dp
        self.n_components = n_components
        self.user_factors = None
        self.item_factors = None
        self.svd = None

    def fit(self):
        # Use TruncatedSVD on user-item matrix
        svd = TruncatedSVD(self.n_components, random_state=42)
        X = self.dp.user_item_matrix
        # fit on dense-ish matrix; for larger datasets use sparse methods
        U = svd.fit_transform(X)
        V = svd.components_.T  # items x components
        self.user_factors = U
        self.item_factors = V
        self.svd = svd
        return self

    def predict_scores_for_user(self, user_id: int, top_k: int = 10) -> List[Tuple[int,float]]:
        idx = self.dp.user_id_to_index.get(user_id)
        if idx is None:
            return []
        u = self.user_factors[idx]
        scores = self.item_factors.dot(u)
        user_row = self.dp.user_item_matrix.getrow(idx).toarray().ravel()
        watched = set(np.where(user_row>0)[0])
        candidates = [(i, float(s)) for i,s in enumerate(scores) if i not in watched]
        candidates.sort(key=lambda x: x[1], reverse=True)
        return [(self.dp.movie_index_to_id[i], score) for i,score in candidates[:top_k]]

In [14]:
# Hybrid Recommender

In [15]:
class HybridRecommender:
    def __init__(self, cf: CollaborativeFiltering, mf: MatrixFactorization, dp: DataProcessor):
        self.cf = cf
        self.mf = mf
        self.dp = dp

    def recommend(self, user_id: int, top_k: int = 10, weights: Dict[str,float] = None) -> List[Tuple[int,float]]:
        # weights: {'item':0.5, 'user':0.2, 'mf':0.3}
        if weights is None:
            weights = {'item':0.4, 'user':0.2, 'mf':0.4}

        scores = {}
        # item-based
        if self.cf.item_sim is None:
            self.cf.fit_item_based('cosine')
        if 'item' in weights and weights['item']>0:
            item_preds = self.cf.predict_item_based_for_user(user_id, top_k=500)
            for mid, s in item_preds:
                scores[mid] = scores.get(mid,0.0) + weights['item']*s
        # user-based
        if self.cf.user_sim is None:
            self.cf.fit_user_based()
        if 'user' in weights and weights['user']>0:
            user_preds = self.cf.predict_user_based_for_user(user_id, top_k=500)
            for mid, s in user_preds:
                scores[mid] = scores.get(mid,0.0) + weights['user']*s
        # mf
        if 'mf' in weights and weights['mf']>0 and self.mf.user_factors is not None:
            mf_preds = self.mf.predict_scores_for_user(user_id, top_k=500)
            for mid, s in mf_preds:
                scores[mid] = scores.get(mid,0.0) + weights['mf']*s

        # fallback for cold-start (new user)
        if not scores:
            # recommend top popular movies
            movie_pop = np.array(self.dp.user_item_matrix.sum(axis=0)).ravel()
            top_idx = np.argsort(movie_pop)[::-1][:top_k]
            return [(self.dp.movie_index_to_id[i], float(movie_pop[i])) for i in top_idx]

        # sorting and ensuring we convert indices to ids
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

In [16]:
# Evaluation metrics

In [18]:
class Evaluator:
    @staticmethod
    def rmse(preds: List[float], truths: List[float]) -> float:
        return math.sqrt(mean_squared_error(truths, preds))

    @staticmethod
    def precision_at_k(recommended: List[int], actual: List[int], k: int) -> float:
        recommended_k = recommended[:k]
        hits = len(set(recommended_k) & set(actual))
        return hits / k

    @staticmethod
    def recall_at_k(recommended: List[int], actual: List[int], k: int) -> float:
        recommended_k = recommended[:k]
        hits = len(set(recommended_k) & set(actual))
        return hits / max(1, len(actual))

In [19]:
# Simple demo / main

In [21]:
!wget -nc http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -n ml-100k.zip

if __name__ == "__main__":
    print("Loading data and building models (this may take a few seconds)...")
    dp = DataProcessor().load().preprocess(min_user_ratings=5, min_movie_ratings=5)

    cf = CollaborativeFiltering(dp)
    cf.fit_item_based('cosine').fit_user_based()

    mf = MatrixFactorization(dp, n_components=20).fit()

    hybrid = HybridRecommender(cf, mf, dp)

    # Example: recommend for user 10 (if present in dataset)
    example_user = dp.user_index_to_id[0]  # first user in filtered data
    print(f"Example user id: {example_user}")
    recs = hybrid.recommend(example_user, top_k=10)
    print("Top recommendations (movie_id, score):")
    for m,score in recs:
        title = dp.items.set_index('movie_id').loc[m]['title'] if m in set(dp.items['movie_id']) else 'Unknown'
        print(f"{m}\t{title}\t{score:.4f}")

    print('\nDone.\n')

--2026-03-05 12:43:29--  http://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k.zip [following]
--2026-03-05 12:43:30--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  2.85MB/s    in 1.6s    

2026-03-05 12:43:32 (2.85 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml